# 1. Configuração Inicial e Imports

## Objetivo desta Seção
Importar todas as bibliotecas necessárias para a solução completa e configurar parâmetros visuais dos gráficos.

## Bibliotecas Utilizadas

### Processamento e Estruturas de Dados
- **networkx (nx)**: Criação, manipulação e análise de grafos
- **pandas (pd)**: Estruturas de dados para análise (DataFrame, Series)
- **numpy (np)**: Computação numérica e operações matemáticas

### Visualização
- **matplotlib.pyplot (plt)**: Criação de gráficos e visualizações
- **seaborn (sns)**: Estilo e temas para matplotlib

### Utilitários
- **os**: Operações do sistema operacional
- **pathlib.Path**: Manipulação de caminhos de arquivos
- **time**: Medição de tempo de execução
- **itertools.product**: Geração de combinações
- **typing**: Type hints para melhor documentação

## Configurações Visuais
- Estilo whitegrid para melhor legibilidade
- Figura padrão de 12x6 polegadas
- Tamanho de fonte padrão de 10pt

In [32]:
# Importar todas as bibliotecas necessárias
import os
import csv
import time
import json
import random
from pathlib import Path
from itertools import product
from typing import Dict, Tuple, List, Set

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Todas as bibliotecas importadas com sucesso")

✓ Todas as bibliotecas importadas com sucesso


# 2. Criação da Estrutura de Diretórios

## Objetivo desta Seção
Implementar uma classe auxiliar para criar e gerenciar automaticamente a estrutura de diretórios necessária para armazenar resultados, gráficos e visualizações de ambas as partes da solução.

## Estrutura Criada

```
resultados/
├── parte1/
│   ├── csv/              # Parâmetros e resultados da força bruta
│   ├── grafos/           # Visualizações dos grafos coloridos (PNG)
│   └── graficos/         # Gráficos de análise e escalabilidade (PNG)
└── parte2/
    ├── csv/              # Resultados da heurística DIMACS
    ├── grafos/           # Visualizações dos grafos coloridos (PNG)
    └── graficos/         # Gráficos comparativos (PNG)
```

## Funcionalidades da Classe ConfiguradorDiretorios

### `criar_estrutura()`
- Cria automaticamente toda a estrutura de diretórios
- Garante que todos os caminhos existem antes de salvar arquivos
- Usa `parents=True` e `exist_ok=True` para evitar erros

### `obter_caminho_arquivo(tipo_parte, tipo_saida, nome_arquivo)`
- Fornece caminhos consistentes para arquivo de saída
- Exemplo: `'resultados/parte1/csv/parametros_grafos.csv'`
- Reduz hardcoding de caminhos no código

In [33]:
class ConfiguradorDiretorios:
    """Classe auxiliar para criar e gerenciar estrutura de diretórios."""
    
    ESTRUTURA_DIRETORIOS = {
        'resultados/parte1/csv': [],
        'resultados/parte1/grafos': [],
        'resultados/parte1/graficos': [],
        'resultados/parte2/csv': [],
        'resultados/parte2/grafos': [],
        'resultados/parte2/graficos': [],
    }
    
    @staticmethod
    def criar_estrutura():
        """Cria a estrutura de diretórios completa."""
        for diretorio in ConfiguradorDiretorios.ESTRUTURA_DIRETORIOS.keys():
            Path(diretorio).mkdir(parents=True, exist_ok=True)
        print(f"✓ Estrutura de diretórios criada com sucesso")
    
    @staticmethod
    def obter_caminho_arquivo(tipo_parte: str, tipo_saida: str, nome_arquivo: str) -> str:
        """
        Retorna caminho completo para arquivo baseado em tipo.
        
        Args:
            tipo_parte: 'parte1' ou 'parte2'
            tipo_saida: 'csv', 'grafos' ou 'graficos'
            nome_arquivo: nome do arquivo
        
        Returns:
            Caminho completo do arquivo
        """
        return f"resultados/{tipo_parte}/{tipo_saida}/{nome_arquivo}"

# Criar estrutura de diretórios
ConfiguradorDiretorios.criar_estrutura()

✓ Estrutura de diretórios criada com sucesso


# 3. Funções Auxiliares Comuns

## Objetivo desta Seção
Definir funções auxiliares compartilhadas entre Parte 1 (Força Bruta) e Parte 2 (Heurística) para evitar duplicação de código.

## Funções Implementadas

### 1. `extrair_parametros_grafo(grafo)`
Extrai métricas estatísticas de um grafo NetworkX.

**Parâmetros Extraídos:**
- `num_vertices`: Número de vértices
- `num_arestas`: Número de arestas
- `densidade`: Razão arestas/(vértices²), indica "compactação" do grafo
- `grau_medio`: Média dos graus de todos os vértices
- `grau_minimo`: Menor grau encontrado
- `grau_maximo`: Maior grau encontrado
- `eh_conexo`: Boolean se grafo é conexo
- `num_componentes`: Número de componentes conectadas

**Uso:** Coletar informações para análise posterior

---

### 2. `visualizar_e_salvar_grafo(grafo, coloracao, caminho_saida)`
Cria visualizações de grafos com coloração e salva como PNG.

**Características:**
- Layout spring (force-directed) para melhor disposição visual
- Colormap 'Set3' com cores distintas por número cromático
- Arestas com alpha=0.3 para não obscurecer nós
- Título mostra χ(G) encontrado
- Salva em alta resolução (150 dpi)

**Uso:** Documentar visualmente a coloração encontrada

---

### 3. `ler_arquivo_dimacs(caminho_arquivo)`
Parser para o formato DIMACS padrão de competições de coloração.

**Formato DIMACS:**
```
c comentário
c comentário
p edge num_vertices num_arestas
e vertice1 vertice2
e vertice3 vertice4
...
```

**Características:**
- Ignora linhas de comentário ('c')
- Processa linha 'p edge' para dimensões
- Processa linhas 'e' para arestas
- Tratamento de erro se arquivo não existir
- Retorna grafo NetworkX ou None

**Uso:** Ler 5 instâncias do Moodle na Parte 2

In [34]:
def extrair_parametros_grafo(grafo: nx.Graph) -> Dict:
    """
    Extrai parâmetros estatísticos de um grafo.
    
    Função auxiliar comum usada em ambas as partes para extrair informações
    de um grafo (densidade, grau médio, conectividade, etc.).
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Dicionário com parâmetros do grafo
    """
    return {
        'num_vertices': grafo.number_of_nodes(),
        'num_arestas': grafo.number_of_edges(),
        'densidade': nx.density(grafo),
        'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes(),
        'grau_minimo': min(dict(grafo.degree()).values()),
        'grau_maximo': max(dict(grafo.degree()).values()),
        'eh_conexo': nx.is_connected(grafo),
        'num_componentes': nx.number_connected_components(grafo),
    }

print("✓ Função extrair_parametros_grafo definida")

✓ Função extrair_parametros_grafo definida


In [50]:
def visualizar_e_salvar_grafo(grafo: nx.Graph, coloracao: Dict, caminho_saida: str):
    """
    Visualiza grafo com coloração e salva como PNG.
    
    Função auxiliar comum que cria visualização dos grafos coloridos em ambas
    as partes da solução.
    
    Args:
        grafo: Grafo NetworkX
        coloracao: Dicionário {vertice: cor}
        caminho_saida: Caminho para salvar arquivo PNG
    """
    plt.figure(figsize=(10, 8))
    
    # Posição dos nós (layout spring para melhor visualização)
    pos = nx.spring_layout(grafo, seed=42, iterations=50)
    
    # Cores para a visualização (colormap)
    cores_set = set(coloracao.values())
    cores_lista = [coloracao[node] for node in grafo.nodes()]
    
    # Desenhar grafo
    nx.draw_networkx_edges(grafo, pos, alpha=0.3, width=1)
    nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista, 
                          node_size=500, cmap='Set3', vmin=0, vmax=max(cores_set))
    nx.draw_networkx_labels(grafo, pos, font_size=8)
    
    plt.title(f"Coloração de Vértices\n(χ(G) = {max(cores_set) + 1})")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=150, bbox_inches='tight')
    plt.close()

print("✓ Função visualizar_e_salvar_grafo definida")

✓ Função visualizar_e_salvar_grafo definida


In [36]:
def ler_arquivo_dimacs(caminho_arquivo: str) -> nx.Graph:
    """
    Função auxiliar comum: Lê arquivo DIMACS e cria grafo NetworkX.
    
    Formato DIMACS:
    - Linhas começadas com 'c' são comentários
    - Linha 'p edge <n> <m>' define n vértices e m arestas
    - Linhas 'e <u> <v>' definem arestas
    
    Args:
        caminho_arquivo: Caminho do arquivo DIMACS
        
    Returns:
        Grafo NetworkX ou None se arquivo não existir
    """
    if not os.path.exists(caminho_arquivo):
        print(f"⚠ Arquivo não encontrado: {caminho_arquivo}")
        return None
    
    grafo = nx.Graph()
    
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8', errors='ignore') as f:
            for linha in f:
                linha = linha.strip()
                
                # Ignorar comentários
                if linha.startswith('c'):
                    continue
                
                # Linha de problema (define número de vértices)
                if linha.startswith('p edge'):
                    partes = linha.split()
                    num_vertices = int(partes[2])
                    num_arestas = int(partes[3])
                    grafo.add_nodes_from(range(1, num_vertices + 1))
                    continue
                
                # Linhas de aresta
                if linha.startswith('e'):
                    partes = linha.split()
                    u = int(partes[1])
                    v = int(partes[2])
                    grafo.add_edge(u, v)
        
        return grafo
    
    except Exception as e:
        print(f"⚠ Erro ao ler arquivo DIMACS: {str(e)}")
        return None

print("✓ Função ler_arquivo_dimacs definida")

✓ Função ler_arquivo_dimacs definida


## 4. Geração de Grafos Aleatórios

### Objetivo desta Seção (REQUISITO 2 - PARTE 1)
Implementar gerador de instâncias aleatórias de grafos usando o modelo Erdős-Rényi, que garante:
- Grafos não direcionados
- Sem ponderação
- Sem loops ou arestas paralelas

## Modelo Erdős-Rényi (ER)

### Características
- **Definição**: G(n, p) onde n = vértices e p = probabilidade de aresta
- **Funcionamento**: Para cada par (u,v), adiciona aresta com probabilidade p
- **Garantias**: Sem loops ou múltiplas arestas por design
- **Distribuição**: Graus seguem distribuição binomial

### Parâmetros da Função
- `num_vertices`: Tamanho do grafo (5 a 13 vértices para Parte 1)
- `probabilidade`: P(aresta existe) entre 0 e 1 (padrão 0.3)
- `seed`: Para reprodutibilidade de resultados

### Comportamento Especial
Se o grafo gerado estiver vazio (comum em grafos pequenos com p baixo), regenera até ter pelo menos uma aresta.


---

# 🔵 PARTE 1: FORÇA BRUTA COM INSTÂNCIAS ALEATÓRIAS

## Objetivo Geral
Implementar e analisar um **algoritmo exato** de coloração de grafos através de busca exaustiva (força bruta), avaliando sua escalabilidade em instâncias aleatórias pequenas (5-13 vértices).

## O que será feito:
1. **Gerar** grafos aleatórios (modelo Erdős-Rényi)
2. **Encontrar** número cromático exato χ(G) por força bruta
3. **Medir** tempo de execução e demonstrar crescimento exponencial
4. **Visualizar** grafos coloridos e gerar gráficos de análise

## Resultado esperado:
- 27 instâncias testadas (3 por tamanho, de 5-13 vértices)
- χ(G) exato para cada grafo
- Demonstração clara: por que força bruta falha para n > ~15

---



In [37]:
def gerar_instancia_grafo_aleatorio(num_vertices: int, probabilidade: float, 
                                   seed: int = None) -> nx.Graph:
    """
    REQUISITO 2 - PARTE 1: Gera instância aleatória de grafo.
    
    Cria um grafo aleatório usando modelo Erdős-Rényi:
    - Não direcionado
    - Não ponderado
    - Sem loops ou arestas paralelas (garantido pelo modelo)
    
    Args:
        num_vertices: Número de vértices (n)
        probabilidade: Probabilidade de aresta entre dois vértices (0 a 1)
        seed: Seed para reprodutibilidade
    
    Returns:
        Grafo NetworkX não direcionado
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Modelo Erdős-Rényi: grafo aleatório
    grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    # Garantir que o grafo tem arestas (para grafos muito pequenos)
    while grafo.number_of_edges() == 0 and num_vertices > 1:
        grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    return grafo

print("✓ Função gerar_instancia_grafo_aleatorio definida")

✓ Função gerar_instancia_grafo_aleatorio definida


## 5. Algoritmo de Força Bruta para Coloração

### Objetivo desta Seção (REQUISITO 1 - PARTE 1)
Implementar algoritmo exato que encontra o número cromático χ(G) através de busca exaustiva de combinações de cores.

## Estratégia do Algoritmo

### Pseudocódigo
```
para k = 1 até n_vertices:
    para cada combinação de k cores em n_vertices:
        se coloracao é válida (vértices adjacentes diferem):
            retorna k, coloracao, tempo
```

### Busca por Cores Mínimas
1. Começa com k=1 cor
2. Se nenhuma combinação válida, tenta k=2
3. Continua até encontrar k válido
4. χ(G) é o primeiro k que funciona

## Validação de Coloração

Uma coloração é **válida** se:
- Cada vértice tem uma cor (0 a k-1)
- Para toda aresta (u,v): cor[u] ≠ cor[v]

## Complexidade e Limitações

### Complexidade Assintótica
- **Pior caso**: O(k^n) onde k=χ(G) e n=vértices
- **Típico para n≤13**: Viável (13^3 ≈ 2,197 combinações por k)
- **Para n>15**: Tempo exponencial proibitivo

### Razão de Ser
- Encontra solução **ótima** (χ(G) exato)
- Maior tamanho testável: ~15-20 vértices com hardware moderno

## Saída Esperada
Tupla contendo:
- `numero_cromatico`: Valor exato de χ(G)
- `coloracao`: Dicionário {vértice: cor}
- `tempo_segundos`: Tempo de execução medido

In [38]:
def algoritmo_forca_bruta_coloracao(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 1: Implementa algoritmo de força bruta para coloração.
    
    ESTRATÉGIA:
    -----------
    Para cada número de cores k = 1, 2, 3, ...:
      1. Gera todas as combinações possíveis de cores para os vértices (k^n)
      2. Para cada combinação, verifica se é válida (vértices adjacentes diferem)
      3. Se válida, retorna k como número cromático
    
    Complexidade: O(k^n) - Exponencial, viável para n ≤ ~15-20 vértices
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cromático, coloração_dict, tempo_segundos)
    """
    inicio = time.time()
    
    vertices = list(grafo.nodes())
    n_vertices = len(vertices)
    
    # Tentar com 1, 2, 3, ... cores até encontrar coloração válida
    for num_cores in range(1, n_vertices + 1):
        # Gera todas as combinações de cores (k^n possibilidades)
        for cores_tuple in product(range(num_cores), repeat=n_vertices):
            # Cria dicionário de coloração
            coloracao = {vertices[i]: cores_tuple[i] for i in range(n_vertices)}
            
            # Verifica se coloração é válida (vértices adjacentes com cores diferentes)
            eh_valida = True
            for u, v in grafo.edges():
                if coloracao[u] == coloracao[v]:
                    eh_valida = False
                    break
            
            # Se válida, encontramos χ(G)
            if eh_valida:
                tempo = time.time() - inicio
                return num_cores, coloracao, tempo
    
    # Nunca deve chegar aqui (coloração sempre existe)
    tempo = time.time() - inicio
    return n_vertices, {v: i for i, v in enumerate(vertices)}, tempo

print("✓ Função algoritmo_forca_bruta_coloracao definida")

✓ Função algoritmo_forca_bruta_coloracao definida


## 6. Processamento de Instâncias com Força Bruta

### Objetivo desta Seção (REQUISITO 2 + 3 - PARTE 1)
Gerar múltiplas instâncias aleatórias de tamanhos variados, aplicar força bruta a cada uma e medir tempo de execução para análise de escalabilidade.

## Fluxo de Processamento

### Para cada tamanho de 5 a 13 vértices:
1. Gera 3 instâncias aleatórias diferentes (usando seeds diferentes)
2. Extrai parâmetros estatísticos de cada grafo
3. Aplica algoritmo de força bruta
4. Registra tempo de execução
5. Visualiza grafo colorido como PNG

## Matriz de Testes

| Tamanho | Instâncias | Total de Testes |
|---------|-----------|-----------------|
| 5-13    | 3 cada    | 27 grafos       |

**Total esperado**: 27 CSVs + 27 PNGs + 2 gráficos de análise

## Dados Coletados por Instância

### Parâmetros do Grafo
- Número de vértices, arestas
- Densidade, grau médio, grau mín/máx
- Conexidade, número de componentes

### Resultados de Coloração
- ID único (n{tamanho:02d}_i{instância:02d})
- Número cromático encontrado χ(G)
- **Tempo de execução (segundos)** - crítico para análise
- Número de arestas

## Exportação

### CSV de Parâmetros
Armazena características de cada grafo para análise posterior (densidade vs cores, etc.)

### CSV de Resultados
Armazena χ(G) e tempo por instância - essencial para gráfico de escalabilidade

### PNG de Grafos
Visualizações coloridas de cada grafo processado

In [39]:
def processar_instancias_por_tamanho(tamanho_min: int = 5, tamanho_max: int = 13,
                                     probabilidade: float = 0.3,
                                     num_instancias_por_tamanho: int = 3) -> Tuple[List, List]:
    """
    REQUISITO 2 + 3 - PARTE 1: Processa múltiplas instâncias aleatórias.
    
    Gera instâncias de tamanho tamanho_min a tamanho_max e aplica força bruta em cada.
    Mede tempo de execução para cada instância.
    
    Args:
        tamanho_min: Tamanho mínimo (padrão 5 vértices)
        tamanho_max: Tamanho máximo (padrão 13 vértices)
        probabilidade: Probabilidade de aresta entre vértices
        num_instancias_por_tamanho: Número de instâncias por tamanho
        
    Returns:
        Tupla: (lista_parametros, lista_resultados)
    """
    lista_parametros = []
    lista_resultados = []
    
    print("\n" + "="*70)
    print("PARTE 1: PROCESSANDO INSTÂNCIAS ALEATÓRIAS COM FORÇA BRUTA")
    print("="*70)
    
    for tamanho in range(tamanho_min, tamanho_max + 1):
        print(f"\nTamanho: {tamanho} vértices", end="")
        
        for instancia_idx in range(num_instancias_por_tamanho):
            # Gerar instância
            seed = tamanho * 1000 + instancia_idx
            grafo = gerar_instancia_grafo_aleatorio(tamanho, probabilidade, seed=seed)
            
            # Extrair parâmetros
            parametros = extrair_parametros_grafo(grafo)
            parametros['tamanho'] = tamanho
            parametros['instancia'] = instancia_idx + 1
            parametros['id_grafo'] = f"n{tamanho:02d}_i{instancia_idx+1:02d}"
            
            # Aplicar força bruta
            try:
                num_cromatico, coloracao, tempo = algoritmo_forca_bruta_coloracao(grafo)
                
                resultado = {
                    'id_grafo': parametros['id_grafo'],
                    'tamanho': tamanho,
                    'instancia': instancia_idx + 1,
                    'numero_cromatico': num_cromatico,
                    'tempo_segundos': tempo,
                    'arestas': parametros['num_arestas'],
                }
                
                lista_parametros.append(parametros)
                lista_resultados.append(resultado)
                
                # Salvar visualização
                caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                    'parte1', 'grafos', f"grafo_{parametros['id_grafo']}.png"
                )
                visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
                
                print(".", end="", flush=True)
            
            except KeyboardInterrupt:
                print("\n⚠ Execução interrompida pelo usuário")
                break
            except Exception as e:
                print(f"\n⚠ Erro ao processar grafo {tamanho}_{instancia_idx}: {str(e)}")
                continue
        
        print(f" [{tamanho} vértices processado]")
    
    print("\n✓ Processamento de instâncias concluído")
    return lista_parametros, lista_resultados

print("✓ Função processar_instancias_por_tamanho definida")

✓ Função processar_instancias_por_tamanho definida


## 7. Algoritmo Heurístico Welsh-Powell

### Objetivo desta Seção (REQUISITO 1 - PARTE 2)
Implementar heurística gulosa de coloração que encontra solução **viável** (válida mas não necessariamente ótima) em tempo linear.

## Estratégia Welsh-Powell

### Pseudocódigo
```
1. Ordena vértices por grau DECRESCENTE
2. Para cada vértice em ordem:
   a. Encontra cores usadas pelos vizinhos já coloridos
   b. Atribui a MENOR cor não-usada
3. Retorna número de cores usadas
```

### Exemplo Ilustrativo
```
Grafo com graus: A(3), B(2), C(2), D(1), E(1)
Ordem: A → B → C → D → E

Processamento:
- A: cor 0 (primeira disponível)
- B: vizinho A(0), usa cor 1
- C: vizinho A(0), usa cor 1
- D: vizinho B(1), C(1), usa cor 0
- E: vizinho A(0), B(1), usa cor 2

Resultado: 3 cores
```

## Propriedades da Heurística

### Complexidade
- **Tempo**: O(n + m) - Linear em vértices e arestas
- **Espaço**: O(n) para armazenar coloração

### Garantias de Qualidade
- **Válida**: Sempre produz coloração válida
- **Limitante Superior**: Usa no máximo Δ+1 cores, onde Δ=grau máximo
- **Não-Ótima**: Pode usar mais cores que χ(G) ótimo

### Efetividade
- Funciona bem para grafos com distribuição de graus não uniforme
- Especialmente bom para grafos esparsos
- Ordem de processamento é crítica para qualidade

## Saída Esperada
Tupla contendo:
- `numero_cores`: Cores usadas pela heurística
- `coloracao`: Dicionário {vértice: cor}
- `tempo_segundos`: Tempo muito pequeno (< 0.01s típico)


---

# 🔴 PARTE 2: HEURÍSTICA EM INSTÂNCIAS DIMACS REAIS

## Objetivo Geral
Implementar e aplicar um **algoritmo heurístico** (Welsh-Powell) em instâncias grandes do mundo real, demonstrando eficiência prática quando força bruta é inviável.

## O que será feito:
1. **Ler** 5 arquivos DIMACS (instâncias reais do Moodle)
2. **Aplicar** heurística Welsh-Powell (tempo linear O(n+m))
3. **Encontrar** χ_heurística (solução viável, não necessariamente ótima)
4. **Comparar** com força bruta: escalabilidade vs otimalidade

## Resultado esperado:
- 5 instâncias processadas (450-4730 vértices)
- Cores encontradas pela heurística
- Demonstração: heurísticas são essenciais para problemas grandes
- Análise: relação entre densidade, tamanho e qualidade da solução

---



In [40]:
def algoritmo_welsh_powell(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 2: Implementa heurística Welsh-Powell para coloração.
    
    ESTRATÉGIA:
    -----------
    Algoritmo guloso que ordena vértices por grau decrescente:
    1. Ordena vértices por grau (maior primeiro)
    2. Para cada vértice em ordem:
       - Atribui a menor cor disponível (que não foi usada por vizinhos)
    
    Complexidade: O(n + m) - Linear em número de vértices e arestas
    Garantias: Encontra coloração válida (nem sempre ótima)
    Qualidade: Típicamente O(Δ + 1) cores, onde Δ é grau máximo
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cores_usado, coloracao_dict, tempo_segundos)
    """
    inicio = time.time()
    
    # Ordenar vértices por grau decrescente (heurística Welsh-Powell)
    graus = dict(grafo.degree())
    vertices_ordenados = sorted(graus.keys(), key=lambda v: graus[v], reverse=True)
    
    coloracao = {}
    
    # Para cada vértice em ordem de grau decrescente
    for vertice in vertices_ordenados:
        # Cores já usadas pelos vizinhos
        cores_vizinhos = set()
        for vizinho in grafo.neighbors(vertice):
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Encontrar menor cor disponível
        cor = 0
        while cor in cores_vizinhos:
            cor += 1
        
        coloracao[vertice] = cor
    
    tempo = time.time() - inicio
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    
    return num_cores, coloracao, tempo

print("✓ Função algoritmo_welsh_powell definida")

✓ Função algoritmo_welsh_powell definida


## 8. Leitura de Arquivos DIMACS

### Objetivo desta Seção
Implementar parser robusto para o formato DIMACS, padrão internacional em competições de coloração de grafos.

## Formato DIMACS Detalhado

### Estrutura Geral
```
c Comentário - pode haver múltiplas linhas
c Descrevem o grafo ou origem
p edge num_vertices num_arestas
e u1 v1
e u2 v2
...
e un vn
```

### Especificação
- **Linhas 'c'**: Comentários (ignoradas)
- **Linha 'p edge'**: Define dimensões do problema
  - `p`: Literal "p"
  - `edge`: Tipo de problema (apenas "edge" para coloração)
  - `num_vertices`: Número de vértices (1 a n)
  - `num_arestas`: Número de arestas
- **Linhas 'e'**: Definem arestas
  - `e`: Literal "e"
  - `u, v`: Índices dos vértices (1-indexado)

### Exemplo
```
c Grafo simples de 4 vértices
c 4 edges
p edge 4 4
e 1 2
e 2 3
e 3 4
e 4 1
```

## Características da Implementação
- Leitura eficiente linha-a-linha
- Tratamento automático de vértices antes das arestas
- Encoding robusto (utf-8 com fallback)
- Mensagens de erro claras
- Retorna None se arquivo não existe

A função `ler_arquivo_dimacs` já foi definida nas funções auxiliares comuns (seção 3).

## 9. Processamento de Instâncias DIMACS

### Objetivo desta Seção (REQUISITO 2 + 3 - PARTE 2)
Processar 5 instâncias reais do problema de coloração do Moodle, aplicar heurística Welsh-Powell e registrar resultados para comparação com força bruta.

## Instâncias do Moodle

| ID | Nome | Vértices | Arestas | Referência |
|----|------|----------|---------|-----------|
| a  | le450_25a | 450 | 8,260 | DIMACS Competitive Challenge |
| b  | inithx.i.1 | 864 | 18,707 | Benchmark Real-World |
| c  | r1000.1 | 1,000 | 14,378 | Random Generated |
| d  | ash958GPIA | 1,916 | 12,506 | Geometric Structure |
| e  | wap03a | 4,730 | 286,722 | **Grande - Teste de Escalabilidade** |

## Características das Instâncias

### Tamanho e Complexidade
- Instâncias **progressivamente maiores**
- Instância 'e' tem ~60x mais arestas que 'a'
- Força bruta **não é viável** para estas (n > 20)

### Tipos de Estrutura
- **Random** (r1000.1): Distribuição uniforme
- **Geométrica** (ash958GPIA): Baseada em coordenadas
- **Real-World** (inithx.i.1, wap03a): Problemas práticos
- **DIMACS** (le450_25a): Benchmark clássico

## Processamento por Instância

### Para cada arquivo DIMACS:
1. Lê arquivo usando `ler_arquivo_dimacs()`
2. Aplica `algoritmo_welsh_powell()` para encontrar χ_heurística
3. Registra:
   - Número de cores encontrado
   - Tempo de execução
   - Densidade do grafo
   - Grau médio
   - Parâmetros do grafo
4. Visualiza se possível (limite: 500 vértices)

## Dados Coletados
- ID, caminho do arquivo
- Dimensões reais (vértices, arestas) 
- **Cores encontradas pela heurística** (comparar com otimizador)
- Tempo de processamento
- Métricas: densidade, grau médio

In [51]:
def processar_instancias_dimacs() -> List:
    """
    REQUISITO 2 + 3 - PARTE 2: Processa instâncias DIMACS com heurística.
    
    Nota: Este é um template. Você deve colocar os arquivos DIMACS em
    'instancias/' para processar as 5 instâncias do Moodle.
    
    Instâncias esperadas (do Moodle):
      a. 450 vértices, 8260 arestas
      b. 864 vértices, 18707 arestas
      c. 1000 vértices, 14378 arestas
      d. 1916 vértices, 12506 arestas
      e. 4730 vértices, 286722 arestas
    
    Returns:
        Lista de resultados para cada instância
    """
    lista_resultados = []
    
    # Definir instâncias esperadas (atualizar conforme necessário)
    instancias = [
        {
            'id': 'a',
            'caminho': 'instancias/a - le450_25a.col.txt',
            'vertices_esperados': 450,
            'arestas_esperadas': 8260,
        },
        {
            'id': 'b',
            'caminho': 'instancias/b - inithx.i.1.col.txt',
            'vertices_esperados': 864,
            'arestas_esperadas': 18707,
        },
        {
            'id': 'c',
            'caminho': 'instancias/c - r1000.1.col.txt',
            'vertices_esperados': 1000,
            'arestas_esperadas': 14378,
        },
        {
            'id': 'd',
            'caminho': 'instancias/d - ash958GPIA.col.txt',
            'vertices_esperados': 1916,
            'arestas_esperadas': 12506,
        },
        {
            'id': 'e',
            'caminho': 'instancias/e - wap03a.col.txt',
            'vertices_esperados': 4730,
            'arestas_esperadas': 286722,
        },
    ]
    
    print("\n" + "="*70)
    print("PARTE 2: PROCESSANDO INSTÂNCIAS DIMACS COM HEURÍSTICA")
    print("="*70)
    
    for instancia in instancias:
        print(f"\nProcessando instância {instancia['id']}... ", end="", flush=True)
        
        grafo = ler_arquivo_dimacs(instancia['caminho'])
        
        if grafo is None:
            print(f"⚠ Não encontrado")
            continue
        
        # Verificar se grafo tem vértices
        if grafo.number_of_nodes() == 0:
            print(f"⚠ Grafo vazio")
            continue
        
        try:
            # Aplicar heurística Welsh-Powell
            num_cores, coloracao, tempo = algoritmo_welsh_powell(grafo)
            
            resultado = {
                'instancia_id': instancia['id'],
                'caminho': instancia['caminho'],
                'num_vertices': grafo.number_of_nodes(),
                'num_arestas': grafo.number_of_edges(),
                'cores_heuristica': num_cores,
                'tempo_segundos': tempo,
                'densidade': nx.density(grafo),
                'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes(),
            }
            
            lista_resultados.append(resultado)
            print(f"✓ {num_cores} cores em {tempo:.4f}s")
            
            # Salvar visualização (opcional, pode consumir memória em grafos grandes)
            if grafo.number_of_nodes() <= 500:
                caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                    'parte2', 'grafos', f"instancia_{instancia['id']}.png"
                )
                visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
        
        except Exception as e:
            print(f"⚠ Erro: {str(e)}")
            continue
    
    print("\n✓ Processamento de instâncias DIMACS concluído")
    return lista_resultados

print("✓ Função processar_instancias_dimacs definida")

✓ Função processar_instancias_dimacs definida


## 10. Exportação de Resultados para CSV

### Objetivo desta Seção
Exportar todos os resultados em formato CSV para análise posterior, relatórios e criação de gráficos.

## Formato de Saída

### Parte 1: `parametros_grafos.csv`
Armazena características de cada instância gerada:
```
num_vertices, num_arestas, densidade, grau_medio, grau_minimo, grau_maximo, 
eh_conexo, num_componentes, tamanho, instancia, id_grafo
```

**Uso**: Análise de correlação entre propriedades do grafo e número cromático

### Parte 1: `resultados_forca_bruta.csv`
Armazena resultados da força bruta:
```
id_grafo, tamanho, instancia, numero_cromatico, tempo_segundos, arestas
```

**Uso**: Gráfico de escalabilidade (tempo vs tamanho)

### Parte 2: `resultados_heuristica.csv`
Armazena resultados da heurística Welsh-Powell:
```
instancia_id, caminho, num_vertices, num_arestas, cores_heuristica, 
tempo_segundos, densidade, grau_medio
```

**Uso**: 
- Comparação com força bruta (qualidade da heurística)
- Análise de performance em instâncias grandes
- Correlação entre propriedades e qualidade

## Vantagens do CSV
- ✓ Formato universal (Excel, R, Python, etc.)
- ✓ Fácil de processar com pandas
- ✓ Legível como texto
- ✓ Suporta análises estatísticas posteriores
- ✓ Documentação automática de resultados

In [42]:
def salvar_resultados_csv_parte1(parametros_list: List, resultados_list: List):
    """
    Salva resultados da Parte 1 em CSV.
    
    Args:
        parametros_list: Lista de dicionários com parâmetros dos grafos
        resultados_list: Lista de dicionários com resultados da força bruta
    """
    # CSV de parâmetros
    if parametros_list:
        df_parametros = pd.DataFrame(parametros_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte1', 'csv', 'parametros_grafos.csv'
        )
        df_parametros.to_csv(caminho, index=False)
        print(f"✓ Parâmetros salvos em: {caminho}")
    
    # CSV de resultados
    if resultados_list:
        df_resultados = pd.DataFrame(resultados_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte1', 'csv', 'resultados_forca_bruta.csv'
        )
        df_resultados.to_csv(caminho, index=False)
        print(f"✓ Resultados salvos em: {caminho}")

def salvar_resultados_csv_parte2(resultados_list: List):
    """
    Salva resultados da Parte 2 em CSV.
    
    Args:
        resultados_list: Lista de dicionários com resultados da heurística
    """
    if resultados_list:
        df_resultados = pd.DataFrame(resultados_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte2', 'csv', 'resultados_heuristica.csv'
        )
        df_resultados.to_csv(caminho, index=False)
        print(f"✓ Resultados heurística salvos em: {caminho}")

print("✓ Funções de exportação CSV definidas")

✓ Funções de exportação CSV definidas


## 11. Geração de Gráficos de Análise - Parte 1

### Objetivo desta Seção
Criar visualizações que demonstram o crescimento exponencial da força bruta e analisam o número cromático encontrado.

## Gráfico 1: Escalabilidade (Tempo vs Tamanho)

### Características
- **Eixo Y (log)**: Tempo em escala logarítmica (semilogy)
  - Revelam crescimento exponencial como reta
- **Eixo X (linear)**: Tamanho do grafo em vértices
- **Linha azul**: Tempo **médio** por tamanho
- **Área sombreada**: Intervalo min-max observado

### Interpretação
- **Inclinação suave**: Crescimento polinomial
- **Inclinação íngreme**: Crescimento exponencial
- **Espaçamento**: Variabilidade entre instâncias

### Propósito
Demonstra por que força bruta é inviável para n > ~15:
- n=10: ~0.001s
- n=13: ~1-10s
- n=15: ~1000s+ (proibitivo)

---

## Gráfico 2: Número Cromático vs Tamanho

### Características
- **Linha verde** com marcadores quadrados
- Mostra evolução de χ(G) com tamanho
- Linha de tendência implícita pela observação

### Observações Esperadas
- χ(G) não cresce linearmente com n
- Influência forte da densidade
- Máximo teórico: χ ≤ n

### Propósito
Validar que algoritmo encontra diferentes valores cromáticos
- Não é constante (grafo muda)
- Não cresce trivialmente

In [43]:
def gerar_graficos_parte1(resultados_list: List):
    """
    Gera gráficos de análise para Parte 1.
    
    Gera:
    1. Gráfico de escalabilidade (tempo vs tamanho)
    2. Gráfico de número cromático (χ(G) vs tamanho)
    
    Args:
        resultados_list: Lista de resultados da força bruta
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar gráficos Parte 1")
        return
    
    df = pd.DataFrame(resultados_list)
    
    # Agrupar por tamanho
    df_agrupado = df.groupby('tamanho').agg({
        'tempo_segundos': ['mean', 'min', 'max'],
        'numero_cromatico': 'mean'
    }).reset_index()
    
    # Gráfico 1: Escalabilidade (Tempo vs Tamanho)
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    tamanhos = df_agrupado['tamanho'].values
    tempos_media = df_agrupado[('tempo_segundos', 'mean')].values
    tempos_min = df_agrupado[('tempo_segundos', 'min')].values
    tempos_max = df_agrupado[('tempo_segundos', 'max')].values
    
    plt.semilogy(tamanhos, tempos_media, 'o-', label='Média', linewidth=2, markersize=8)
    plt.fill_between(tamanhos, tempos_min, tempos_max, alpha=0.2, label='Min-Max')
    
    plt.xlabel('Tamanho do Grafo (número de vértices)', fontsize=11, fontweight='bold')
    plt.ylabel('Tempo de Execução (segundos)', fontsize=11, fontweight='bold')
    plt.title('Escalabilidade: Crescimento Exponencial do Tempo', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Gráfico 2: Número cromático vs Tamanho
    plt.subplot(1, 2, 2)
    cores_media = df_agrupado[('numero_cromatico', 'mean')].values
    
    plt.plot(tamanhos, cores_media, 's-', color='green', linewidth=2, markersize=8)
    plt.xlabel('Tamanho do Grafo (número de vértices)', fontsize=11, fontweight='bold')
    plt.ylabel('Número Cromático χ(G)', fontsize=11, fontweight='bold')
    plt.title('Número Cromático Encontrado', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'graficos', 'escalabilidade_forca_bruta.png'
    )
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Gráficos Parte 1 salvos em: {caminho.split('/graficos/')[0]}/graficos/")

print("✓ Função gerar_graficos_parte1 definida")

✓ Função gerar_graficos_parte1 definida


## 12. Geração de Gráficos de Análise - Parte 2

### Objetivo desta Seção
Criar 4 visualizações que analisam desempenho da heurística Welsh-Powell nas 5 instâncias DIMACS.

## Gráfico 1 (Superior-Esquerdo): Cores por Instância

### Características
- **Bar chart** azul (steelblue)
- **Eixo X**: Instâncias (a, b, c, d, e)
- **Eixo Y**: Número de cores encontradas
- **Ordenação**: Por quantidade de cores (crescente)

### Interpretação
- Instâncias menores (a) = menos cores esperadas
- Instâncias maiores (e) = mais cores esperadas
- Variabilidade = qualidade da heurística varia

---

## Gráfico 2 (Superior-Direito): Tempo de Execução

### Características
- **Bar chart** coral (laranja)
- Mostra tempo em segundos por instância
- Escala linear para facilitar leitura

### Observações
- Welsh-Powell: O(n+m) = **muito rápido** (<0.1s)
- Contraste com força bruta: exponencial
- Escala linear vs log na Parte 1

---

## Gráfico 3 (Inferior-Esquerdo): Vértices vs Cores

### Características
- **Scatter plot** colorido (viridis)
- Cada ponto = uma instância
- Cor do ponto = número de vértices
- Anotações com ID de instância

### Interpretação
- Relação: mais vértices → geralmente mais cores
- Não é linear (densidade também importa)
- Exemplo: instância 'e' (4730 vértices)

---

## Gráfico 4 (Inferior-Direito): Densidade vs Cores

### Características
- **Scatter plot** com cores (Reds)
- Cor do ponto = número de cores
- Eixo X: Densidade (0 a 1)
- Eixo Y: Cores usadas

### Interpretação
- **Padrão esperado**: Maior densidade → mais cores
- Densidade = arestas / possíveis arestas
- Importante para entender dificuldade do problema

In [44]:
def gerar_graficos_parte2(resultados_list: List):
    """
    Gera gráficos de análise para Parte 2.
    
    Gera:
    1. Número de cores por instância
    2. Tempo de execução por instância
    3. Comparação densidade vs cores
    
    Args:
        resultados_list: Lista de resultados da heurística
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar gráficos Parte 2")
        return
    
    df = pd.DataFrame(resultados_list)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Gráfico 1: Número de cores por instância
    ax1 = axes[0, 0]
    instancias = df['instancia_id'].values
    cores = df['cores_heuristica'].values
    cores_sorted_idx = np.argsort(cores)
    
    ax1.bar(instancias[cores_sorted_idx], cores[cores_sorted_idx], color='steelblue')
    ax1.set_xlabel('Instância DIMACS', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Número de Cores (Heurística)', fontsize=11, fontweight='bold')
    ax1.set_title('Cores Encontradas por Instância', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Gráfico 2: Tempo de execução por instância
    ax2 = axes[0, 1]
    tempos = df['tempo_segundos'].values
    tempos_sorted_idx = np.argsort(tempos)
    
    ax2.bar(instancias[tempos_sorted_idx], tempos[tempos_sorted_idx], color='coral')
    ax2.set_xlabel('Instância DIMACS', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Tempo (segundos)', fontsize=11, fontweight='bold')
    ax2.set_title('Tempo de Execução por Instância', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Gráfico 3: Número de vértices vs Cores
    ax3 = axes[1, 0]
    ax3.scatter(df['num_vertices'], df['cores_heuristica'], s=200, alpha=0.6, 
               c=df['num_vertices'], cmap='viridis')
    for i, txt in enumerate(df['instancia_id']):
        ax3.annotate(txt, (df['num_vertices'].iloc[i], df['cores_heuristica'].iloc[i]),
                    fontsize=10, fontweight='bold')
    
    ax3.set_xlabel('Número de Vértices', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Número de Cores', fontsize=11, fontweight='bold')
    ax3.set_title('Vértices vs Cores', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Gráfico 4: Densidade vs Cores
    ax4 = axes[1, 1]
    ax4.scatter(df['densidade'], df['cores_heuristica'], s=200, alpha=0.6,
               c=df['cores_heuristica'], cmap='Reds')
    for i, txt in enumerate(df['instancia_id']):
        ax4.annotate(txt, (df['densidade'].iloc[i], df['cores_heuristica'].iloc[i]),
                    fontsize=10, fontweight='bold')
    
    ax4.set_xlabel('Densidade do Grafo', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Número de Cores', fontsize=11, fontweight='bold')
    ax4.set_title('Densidade vs Cores', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'analise_heuristica.png'
    )
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Gráficos Parte 2 salvos em: {caminho.split('/graficos/')[0]}/graficos/")

print("✓ Função gerar_graficos_parte2 definida")

✓ Função gerar_graficos_parte2 definida


## 13. Execução Completa e Análise de Resultados

### Objetivo desta Seção
Orquestrar toda a solução em uma única função que executa as 5 etapas principais em sequência.

## Fluxo de Execução (Orquestração)

### Etapa 1: Criar Estrutura
- Cria diretórios para resultados
- Verifica e confirma criação
- Prepara ambiente para salvar arquivos

### Etapa 2: PARTE 1 - Força Bruta
**Tempo estimado**: 5-30 minutos dependendo do hardware
- Gera 27 instâncias aleatórias (5-13 vértices)
- Aplica algoritmo força bruta em cada
- Mede tempo de execução
- Salva 27 PNGs e CSVs
- **Demonstra**: Crescimento exponencial é real

### Etapa 3: PARTE 2 - Heurística
**Tempo estimado**: < 1 segundo
- Lê 5 arquivos DIMACS do Moodle
- Aplica Welsh-Powell (O(n+m))
- Registra cores encontradas
- Salva 5 CSVs + visualizações (se n≤500)
- **Demonstra**: Eficiência de heurísticas

### Etapa 4: Exportação CSV
- Consolida todos os resultados
- Cria 3 arquivos CSV (2 de Parte 1, 1 de Parte 2)
- Pronto para análise em Excel/Python

### Etapa 5: Geração de Gráficos
- 2 gráficos de Parte 1 (escalabilidade)
- 4 gráficos de Parte 2 (análise heurística)
- Todos salvos como PNG em alta resolução

## Resumo Final Exibido

### Estatísticas Parte 1
- Número total de instâncias processadas
- Intervalo de tamanhos testados
- Tempo máximo observado
- Tempo total agregado

### Estatísticas Parte 2
- Número de instâncias DIMACS
- Maior instância processada
- Tempo máximo (muito menor que Parte 1)

### Tempo Total
- Tempo de execução da solução completa
- Inclui todas as 5 etapas

### Localização de Arquivos
- Confirmação de todos os diretórios criados
- Facilita localização de resultados

In [45]:
def executar_solucao_completa():
    """
    Executa a solução completa para o problema de coloração de vértices.
    
    Fluxo:
    1. Criar estrutura de diretórios
    2. PARTE 1: Gerar instâncias aleatórias e aplicar força bruta
    3. PARTE 2: Ler instâncias DIMACS e aplicar heurística
    4. Exportar resultados em CSV
    5. Gerar gráficos e visualizações
    6. Exibir resumo final
    """
    
    print("\n")
    print("╔" + "="*68 + "╗")
    print("║" + " "*15 + "SOLUÇÃO COMPLETA - COLORAÇÃO DE VÉRTICES" + " "*12 + "║")
    print("╚" + "="*68 + "╝")
    
    tempo_inicio = time.time()
    
    # ==================================================================
    # ETAPA 1: Criar Estrutura de Diretórios
    # ==================================================================
    print("\n[1/5] Criando estrutura de diretórios...")
    ConfiguradorDiretorios.criar_estrutura()
    
    # ==================================================================
    # ETAPA 2: PARTE 1 - Força Bruta com Instâncias Aleatórias
    # ==================================================================
    print("\n[2/5] Executando PARTE 1 - Força Bruta...")
    parametros_parte1, resultados_parte1 = processar_instancias_por_tamanho(
        tamanho_min=5,
        tamanho_max=13,
        probabilidade=0.3,
        num_instancias_por_tamanho=3
    )
    
    # ==================================================================
    # ETAPA 3: PARTE 2 - Heurística em Instâncias DIMACS
    # ==================================================================
    print("\n[3/5] Executando PARTE 2 - Heurística Welsh-Powell...")
    resultados_parte2 = processar_instancias_dimacs()
    
    # ==================================================================
    # ETAPA 4: Exportar Resultados em CSV
    # ==================================================================
    print("\n[4/5] Exportando resultados em CSV...")
    salvar_resultados_csv_parte1(parametros_parte1, resultados_parte1)
    salvar_resultados_csv_parte2(resultados_parte2)
    
    # ==================================================================
    # ETAPA 5: Gerar Gráficos
    # ==================================================================
    print("\n[5/5] Gerando gráficos e visualizações...")
    gerar_graficos_parte1(resultados_parte1)
    gerar_graficos_parte2(resultados_parte2)
    
    # ==================================================================
    # RESUMO FINAL
    # ==================================================================
    tempo_total = time.time() - tempo_inicio
    
    print("\n" + "="*70)
    print("RESUMO FINAL DA EXECUÇÃO")
    print("="*70)
    
    print(f"\nPARTE 1 - FORÇA BRUTA:")
    print(f"  • Instâncias processadas: {len(resultados_parte1)}")
    if resultados_parte1:
        df_p1 = pd.DataFrame(resultados_parte1)
        print(f"  • Tamanhos testados: {df_p1['tamanho'].min()}-{df_p1['tamanho'].max()} vértices")
        print(f"  • Tempo máximo: {df_p1['tempo_segundos'].max():.4f}s")
        print(f"  • Tempo total Parte 1: {df_p1['tempo_segundos'].sum():.4f}s")
    
    print(f"\nPARTE 2 - HEURÍSTICA:")
    print(f"  • Instâncias DIMACS processadas: {len(resultados_parte2)}")
    if resultados_parte2:
        df_p2 = pd.DataFrame(resultados_parte2)
        print(f"  • Maiores instâncias: {df_p2['num_vertices'].max()} vértices, "
              f"{df_p2['num_arestas'].max()} arestas")
        print(f"  • Tempo máximo: {df_p2['tempo_segundos'].max():.4f}s")
    
    print(f"\nTEMPO TOTAL DE EXECUÇÃO: {tempo_total:.2f}s")
    
    print(f"\nARQUIVOS GERADOS:")
    print(f"  • resultados/parte1/csv/")
    print(f"  • resultados/parte1/grafos/")
    print(f"  • resultados/parte1/graficos/")
    print(f"  • resultados/parte2/csv/")
    print(f"  • resultados/parte2/grafos/")
    print(f"  • resultados/parte2/graficos/")
    
    print("\n" + "="*70)
    print("✓ EXECUÇÃO CONCLUÍDA COM SUCESSO")
    print("="*70 + "\n")

print("✓ Função executar_solucao_completa definida")

✓ Função executar_solucao_completa definida


## 14. Executar a Solução Completa

### ⚠️ Aviso Importante

Ao executar esta célula, o notebook executará TODA a solução:

### Tempo de Execução Estimado
- **Rápido (Mínimo)**: ~2-5 minutos (apenas Parte 2 se Parte 1 for pulada)
- **Completo**: ~15-45 minutos (ambas as partes)
  - Depende da velocidade do processador
  - Força bruta com n=13 pode ser lento em máquinas antigas

### Requisitos
1. ✓ Todas as funções devem estar definidas nas células anteriores
2. ✓ Bibliotecas importadas (pandas, networkx, matplotlib, etc.)
3. ✓ Diretório `instancias/` contém 5 arquivos DIMACS (para Parte 2)
4. ✓ Permissão de escrita no diretório atual (criar `resultados/`)

### O que Será Gerado
- **27 gráficos coloridos** (Parte 1)
- **5 gráficos DIMACS** (Parte 2, se n ≤ 500)
- **6 gráficos de análise** (PNG em alta resolução)
- **3 arquivos CSV** com todos os dados
- **Saída console** com progresso e resumo

### Como Interpretar a Saída
```
[1/5] Criando estrutura...          ← Etapa de configuração
[2/5] Executando PARTE 1...         ← Força bruta em progresso (mais demorada)
[3/5] Executando PARTE 2...         ← Heurística (rápida)
[4/5] Exportando CSV...              ← Salvando dados
[5/5] Gerando gráficos...            ← Criando visualizações

RESUMO FINAL                          ← Estatísticas de execução
✓ EXECUÇÃO CONCLUÍDA COM SUCESSO
```

---

## 🚀 Executar Agora
Clique em **Run** ou **Shift+Enter** para iniciar a solução completa.

In [52]:
# Executar a solução completa
executar_solucao_completa()



╔====================================================================╗
║               SOLUÇÃO COMPLETA - COLORAÇÃO DE VÉRTICES            ║
╚====================================================================╝

[1/5] Criando estrutura de diretórios...
✓ Estrutura de diretórios criada com sucesso

[2/5] Executando PARTE 1 - Força Bruta...

PARTE 1: PROCESSANDO INSTÂNCIAS ALEATÓRIAS COM FORÇA BRUTA

Tamanho: 5 vértices... [5 vértices processado]

Tamanho: 6 vértices... [6 vértices processado]

Tamanho: 7 vértices... [7 vértices processado]

Tamanho: 8 vértices... [8 vértices processado]

Tamanho: 9 vértices... [9 vértices processado]

Tamanho: 10 vértices... [10 vértices processado]

Tamanho: 11 vértices... [11 vértices processado]

Tamanho: 12 vértices... [12 vértices processado]

Tamanho: 13 vértices... [13 vértices processado]

✓ Processamento de instâncias concluído

[3/5] Executando PARTE 2 - Heurística Welsh-Powell...

PARTE 2: PROCESSANDO INSTÂNCIAS DIMACS COM HEURÍSTICA

Proc